# 00) Data exploration & building `omics.pkl`

The MOFA tools in `src/mofa_tools.py` expect one pre-aligned object:

```python
load_omics_data(data_dir)  # reads  data_dir / "omics.pkl"
```

> Expects a dict with keys `'transcriptomics'`, `'proteomics'`, `'methylation'`,
> and `'meta'` (the subtype labels), all indexed by the same patient IDs in the
> same order.

What we actually have in `data/` is three separate pickles, each a dict of
`{"expr": ..., "meta": ...}`, and the three views cover different, unaligned
patient sets. This notebook inspects them and reconstructs the single aligned
`omics.pkl` the tools expect, without inventing any values.

Run this notebook with the `eccb` kernel.

## 0. Setup

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

VIEWS = ["transcriptomics", "proteomics", "methylation"]
LABEL_COL = "paper_BRCA_Subtype_PAM50"   # PAM50 subtype -> this becomes `meta`

print("DATA_DIR:", DATA_DIR)
print("files:", sorted(p.name for p in DATA_DIR.glob("*.pkl")))

DATA_DIR: c:\Users\elisa\Documents\mcp\ECCB2026_TEST\sessions\session-3-agentic-llm-workflows\data
files: ['methylation.pkl', 'omics.pkl', 'proteomics.pkl', 'transcriptomics.pkl']


## 1. What's inside each view pickle?

Each `<view>.pkl` is a dict with two keys:

- `expr`: a patients × features DataFrame (the actual omics measurements)
- `meta`: a patients × clinical-columns DataFrame (includes the PAM50 subtype)

In [3]:
raw = {name: pd.read_pickle(DATA_DIR / f"{name}.pkl") for name in VIEWS}

for name in VIEWS:
    expr, meta = raw[name]["expr"], raw[name]["meta"]
    print(f"### {name}")
    print(f"  expr : {expr.shape[0]:>5} patients x {expr.shape[1]:>6} features "
          f"| e.g. cols {list(expr.columns[:3])}")
    print(f"  meta : {meta.shape[0]:>5} patients x {meta.shape[1]} cols "
          f"| {list(meta.columns)}")
    print()

### transcriptomics
  expr :  1047 patients x  29995 features | e.g. cols ['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.13']
  meta :  1047 patients x 7 cols | ['patient', 'race', 'gender', 'sample_type', 'paper_BRCA_Subtype_PAM50', 'sizeFactor', 'replaceable']

### proteomics
  expr :   869 patients x    464 features | e.g. cols ['1433BETA', '1433EPSILON', '1433ZETA']
  meta :   869 patients x 5 cols | ['patient', 'race', 'gender', 'sample_type', 'paper_BRCA_Subtype_PAM50']

### methylation
  expr :   780 patients x 200000 features | e.g. cols ['cg11738485', 'cg01893212', 'cg23179456']
  meta :   780 patients x 5 cols | ['patient', 'race', 'gender', 'sample_type', 'paper_BRCA_Subtype_PAM50']



## 2. Subtype labels per view

The `meta` we need is the PAM50 subtype column. Its distribution differs per view because the views cover different patients.

In [4]:
for name in VIEWS:
    counts = raw[name]["meta"][LABEL_COL].value_counts(dropna=False)
    print(f"{name} ({len(raw[name]['meta'])} patients):")
    print(counts.to_string(), "\n")

transcriptomics (1047 patients):
paper_BRCA_Subtype_PAM50
LumA      548
LumB      204
Basal     174
Her2       81
Normal     40 

proteomics (869 patients):
paper_BRCA_Subtype_PAM50
LumA      433
LumB      177
Basal     155
Her2       75
Normal     29 

methylation (780 patients):
paper_BRCA_Subtype_PAM50
LumA      422
LumB      141
Basal     137
Her2       46
Normal     34 



## 3. The views are NOT aligned

`load_omics_data` asserts every view shares the same patient index in the same
order. That is not true of the raw files, different patient sets and
different orderings so we must reconstruct an aligned cohort.

In [5]:
index_by_view = {name: raw[name]["expr"].index for name in VIEWS}

for name in VIEWS:
    print(f"{name:15s}: {len(index_by_view[name])} patients, "
          f"index name = {index_by_view[name].name}")

t, p, m = (set(index_by_view[v]) for v in VIEWS)
print("\npairwise overlap:")
print(f"  transcriptomics & proteomics : {len(t & p)}")
print(f"  transcriptomics & methylation: {len(t & m)}")
print(f"  proteomics      & methylation: {len(p & m)}")
print(f"\nintersection of all three     : {len(t & p & m)}")

transcriptomics: 1047 patients, index name = None
proteomics     : 869 patients, index name = None
methylation    : 780 patients, index name = None

pairwise overlap:
  transcriptomics & proteomics : 840
  transcriptomics & methylation: 745
  proteomics      & methylation: 631

intersection of all three     : 603


## 4. Reconstruct the shared cohort

Take patients present in all three views, drop any without a PAM50 label,
and confirm the label is consistent across the three `meta` tables (a patient
must not be called LumA in one view and Basal in another).

In [6]:
common = sorted(set.intersection(*(set(index_by_view[v]) for v in VIEWS)))
print("patients in all three views:", len(common))

# Subtype label from each view's meta, restricted to the common patients.
labels_wide = pd.DataFrame(
    {name: raw[name]["meta"].loc[common, LABEL_COL] for name in VIEWS}
)

# Consistency: every row should have exactly one unique label across views.
consistent = labels_wide.nunique(axis=1, dropna=True).le(1)
print("label consistent across views:", int(consistent.sum()), "/", len(labels_wide))
assert consistent.all(), "Subtype labels disagree across views for some patients."

subtype = labels_wide[VIEWS[0]]
labeled = subtype.dropna().index.tolist()
print("patients with a PAM50 label :", len(labeled))
print("dropped (missing label)     :", len(common) - len(labeled))

patients in all three views: 603
label consistent across views: 603 / 603
patients with a PAM50 label : 603
dropped (missing label)     : 0


## 5. Assemble the aligned `omics.pkl` dict

Reindex every view's `expr` onto the same labeled-patient index, in the same order, and attach `meta` (the subtype Series).

In [7]:
patient_ids = pd.Index(labeled, name="patient_id")

omics = {}
for name in VIEWS:
    expr = raw[name]["expr"].loc[patient_ids]
    expr.index = expr.index.astype(str)
    omics[name] = expr

meta = subtype.loc[patient_ids].astype(str)
meta.index = meta.index.astype(str)
omics["meta"] = meta

for k, v in omics.items():
    print(f"{k:15s}: {type(v).__name__:9s} {getattr(v, 'shape', None)}")

print("\nfinal subtype distribution:")
print(meta.value_counts().to_string())

transcriptomics: DataFrame (603, 29995)
proteomics     : DataFrame (603, 464)
methylation    : DataFrame (603, 200000)
meta           : Series    (603,)

final subtype distribution:
transcriptomics
LumA      322
LumB      118
Basal      97
Her2       41
Normal     25


## 6. Sanity-check against the loader's assertions

Replicate the exact checks `load_omics_data` performs, so we know the file will load cleanly.

In [8]:
ref_index = omics["meta"].index.astype(str)
for name in VIEWS:
    assert omics[name].index.astype(str).equals(ref_index), f"index mismatch in {name}"
    assert not omics[name].isna().all(axis=None), f"{name} is entirely NaN"
print("All views share the meta index, in order:  OK")
print("Patients:", len(ref_index), "| Views:", VIEWS)

All views share the meta index, in order:  OK
Patients: 603 | Views: ['transcriptomics', 'proteomics', 'methylation']


## 7. Save `omics.pkl`

This is the single file `load_omics_data(DATA_DIR)` will read in the agent notebooks.

In [ ]:
out_path = DATA_DIR / "omics.pkl"
pd.to_pickle(omics, out_path)
print("wrote:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")

# Round-trip check via the actual tool.
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.mofa_tools import load_omics_data

X_omics, y = load_omics_data(DATA_DIR)
print("load_omics_data OK:",
      {k: v.shape for k, v in X_omics.items()}, "| y:", y.shape)

wrote: c:\Users\elisa\Documents\mcp\ECCB2026_TEST\sessions\session-3-agentic-llm-workflows\data\omics.pkl (1115.0 MB)
load_omics_data OK -> {'transcriptomics': (603, 29995), 'proteomics': (603, 464), 'methylation': (603, 200000)} | y: (603,)
